**TransForm Customer Data**

In [0]:
%python
dfCustomers=spark.sql('SELECT * FROM gizmobox_gr.bronze.py_customers')
display(dfCustomers)

In [0]:
%python
dfCustomers=spark.table('gizmobox_gr.bronze.py_customers')
display(dfCustomers)

In [0]:
%python
dfCustomers=spark.read.table('gizmobox_gr.bronze.py_customers')
display(dfCustomers)

**1.Remove Records With Null Data**

In [0]:
%python
dfCustomers=spark.read.table('gizmobox_gr.bronze.py_customers')
dfFilteredData=dfCustomers.filter('customer_id is not null')
display(dfFilteredData)

In [0]:
%python
dfFilteredData=dfCustomers.filter(dfCustomers.customer_id.isNotNull())
display(dfFilteredData)

**2.Remove Records Exact Duplicate Records**

In [0]:
%python
dfCustomersDistinct=dfCustomers.distinct()
display(dfCustomersDistinct)

In [0]:
%python     
dfCustomersDistinct=dfCustomers.dropDuplicates()
display(dfCustomersDistinct)

**3.Remove Duplicate Records Based On Created Timestamp**

In [0]:
%python
from pyspark.sql import functions as f

df_max_ts=dfCustomersDistinct.groupBy('customer_id') \
    .agg(f.max('created_timestamp').alias("max_created_timestamp"))
display(df_max_ts)

In [0]:
%python
df_customers_distinct=dfCustomersDistinct.join(df_max_ts,(dfCustomersDistinct.customer_id==df_max_ts.customer_id & dfCustomersDistinct.created_timestamp==df_max_ts.max_created_timestamp),"inner").select(dfCustomersDistinct["*"])

display(df_customers_distinct) 

**4.CAST The Columns Into Correct Data Types**

In [0]:
%python
df_customers_castDataTypes=(dfCustomersDistinct
                            .select(dfCustomersDistinct.file_path,
                                    dfCustomersDistinct.created_timestamp.cast("timestamp"),
                                    dfCustomersDistinct.customer_id,
                                    dfCustomersDistinct.customer_name,
                                    dfCustomersDistinct.date_of_birth.cast("date"),
                                    dfCustomersDistinct.email,
                                    dfCustomersDistinct.member_since.cast("date"),
                                    dfCustomersDistinct.telephone
                                    )
                            )
display(df_customers_castDataTypes)

**5.Write Transformed Data To Silver Schema**

In [0]:
%python
df_customers_castDataTypes.writeTo("gizmobox_gr.silver.py_customers").createOrReplace();
dfCustomers_Silver= spark.table("gizmobox_gr.silver.py_customers")
display(dfCustomers_Silver)